In [144]:
import pandas as pd

In [145]:
prev_control_totals_xlsx = "_archive/SE/_Source - ControlTotal_SE - 2022-08-31.xlsx"
gpi_employment_by_county_csv = "data/GPI-SE-Data-20251121/Employment_Projections_by_County_by_Industry_Vintage2025_for_WFRC.csv"

# Read in GPI employment data by county

In [146]:
gpi_employment_by_county_df = pd.read_csv(gpi_employment_by_county_csv)

gpi_employment_by_county_df.columns = gpi_employment_by_county_df.columns.str.replace(r"[^0-9A-Za-z]+", "_", regex=True).str.lower()

# Clean Industry names for merging, use lower case and remove non-alphabetic characters
gpi_employment_by_county_df["industry"] = (
    gpi_employment_by_county_df["industry"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z\s]+", "", regex=True)  # keep a–z and spaces
    .str.replace(r"\s+", " ", regex=True)       # collapse multiple spaces
    .str.strip()                                # remove leading/trailing spaces
)

# make sure only county level data is used
gpi_employment_by_county_df = gpi_employment_by_county_df[gpi_employment_by_county_df['areatype']=='County']
gpi_employment_by_county_df.rename(columns={'area':'county'}, inplace=True)
gpi_employment_by_county_df.drop(columns=['areatype'], inplace=True)

gpi_employment_by_county_df

,year,industry,empcount,county,ishistorical
0,2023,accommodation and food services,517.000000,Beaver,1
1,2024,accommodation and food services,481.000000,Beaver,1
2,2025,accommodation and food services,483.522836,Beaver,0
3,2026,accommodation and food services,496.285253,Beaver,0
4,2027,accommodation and food services,497.151040,Beaver,0
...,...,...,...,...,...
29923,2061,wholesale trade,5944.337619,Weber,0
29924,2062,wholesale trade,5977.005576,Weber,0
29925,2063,wholesale trade,6010.644866,Weber,0
29926,2064,wholesale trade,6043.334823,Weber,0


# Get employment lookup from previous control total file 

In [147]:
# open excel file and import worksheet ="Lookup" and range AB4:AL30 with header row
employment_classification_df = pd.read_excel(
    prev_control_totals_xlsx,
    sheet_name="Lookup",
    header=0,         # Row 4 is the header → zero-based index 3
    usecols="AB:AL",  # Restrict to columns AB through AL
    skiprows=lambda x: x < 3 or x > 29  # Keep rows 4–30 only
)
# Add underscores
employment_classification_df.columns = employment_classification_df.columns.str.replace(r"[^0-9A-Za-z]+", "_", regex=True).str.lower()

# Clean Industry names for merging, use lower case and remove non-alphabetic characters
employment_classification_df["gpi_industry"] = (
    employment_classification_df["gpi_industry"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z\s]+", "", regex=True)  # keep a–z and spaces
    .str.replace(r"\s+", " ", regex=True)       # collapse multiple spaces
    .str.strip()                                # remove leading/trailing spaces
)

# remove "State and Local Government" and "total jobs" rows that aren't in GPI data
employment_classification_df = employment_classification_df[
    ~employment_classification_df["gpi_industry"].isin(["state and local government", "total jobs"])
].reset_index(drop=True)

# recode "State Government" and "Local Government" to "GVED" and corresponding IDs
mask = employment_classification_df["gpi_industry"].isin(["state government", "local government"])
employment_classification_df.loc[mask, ["tdm_sector", "tdm_sector_id", "tdm_7_sector", "tdm_7_sector_id"]] = [
    ["GVED", 6, "OTH", 3]
]

# export to processed data folder
employment_classification_df.to_csv("data-processed/employment_classification_lookup.csv", index=False)

employment_classification_df

,gpi_industry,gpi_naics,gopb_sector_id,gopb_sector,tdm_sector_id,tdm_sector,tdm_7_sector_id,tdm_7_sector,acs_sector_id,acs_industry_sectors,dws
0,forestry fishing and hunting,113-115,1,Natural Resources,9,AGRI,4,AGRI,1,"Agriculture, forestry, fishing and hunting, an...",113-114
1,mining,21,2,Mining,10,MING,5,MING,1,"Agriculture, forestry, fishing and hunting, an...",21
2,utilities,22,3,Utilities,4,WSLE,2,IND,6,"Transportation and warehousing, and utilities",22
3,construction,23,4,Construction,11,CONS,6,CONS,2,Construction,23
4,manufacturing,31-33,5,Manufacturing,3,MANU,2,IND,3,Manufacturing,31-33
5,wholesale trade,42,6,Wholesale Trade,4,WSLE,2,IND,4,Wholesale trade,42
6,retail trade,44-45,7,Retail Trade,1,RETL,1,RET,5,Retail trade,44-45
7,transportation and warehousing,48-49,8,Transportation & Warehousing,4,WSLE,2,IND,6,"Transportation and warehousing, and utilities",48-49
8,information,51,9,Information,5,OFFI,3,OTH,7,Information,51
9,finance and insurance,52,10,Finance & Insurance,8,OTHR,3,OTH,8,"Finance and insurance, and real estate and ren...",52


In [148]:
gpi_to_tdm_emp_category_df = employment_classification_df[["gpi_industry", "tdm_sector"]].drop_duplicates()
gpi_to_tdm_emp_category_df.rename(columns={"gpi_industry": "industry"}, inplace=True)

# Clean Industry names for merging, use lower case and remove non-alphabetic characters
gpi_to_tdm_emp_category_df["industry"] = (
    gpi_to_tdm_emp_category_df["industry"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z\s]+", "", regex=True)  # keep a–z and spaces
    .str.replace(r"\s+", " ", regex=True)       # collapse multiple spaces
    .str.strip()                                # remove leading/trailing spaces
)

gpi_to_tdm_emp_category_df

,industry,tdm_sector
0,forestry fishing and hunting,AGRI
1,mining,MING
2,utilities,WSLE
3,construction,CONS
4,manufacturing,MANU
5,wholesale trade,WSLE
6,retail trade,RETL
7,transportation and warehousing,WSLE
8,information,OFFI
9,finance and insurance,OTHR


# Join and aggregate by TDM categories

In [149]:
# check for any industries in GPI employment data not in the lookup
left_only = set(gpi_employment_by_county_df["industry"]) - set(gpi_to_tdm_emp_category_df["industry"])
if left_only==set():
    print("All industries in GPI employment data have a matching TDM category.")

All industries in GPI employment data have a matching TDM category.


In [150]:
# check for any industries in lookup not in GPI employment data
right_only = set(gpi_to_tdm_emp_category_df["industry"]) - set(gpi_employment_by_county_df["industry"])
if right_only==set():
    print("All industries in the lookup have matching GPI employment data.")

All industries in the lookup have matching GPI employment data.


In [151]:
gpi_data_with_tdm_codes_df = pd.merge(gpi_employment_by_county_df, gpi_to_tdm_emp_category_df, on="industry", how="outer")
gpi_data_with_tdm_codes_df

,year,industry,empcount,county,ishistorical,tdm_sector
0,2023,accommodation and food services,517.000000,Beaver,1,FOOD
1,2024,accommodation and food services,481.000000,Beaver,1,FOOD
2,2025,accommodation and food services,483.522836,Beaver,0,FOOD
3,2026,accommodation and food services,496.285253,Beaver,0,FOOD
4,2027,accommodation and food services,497.151040,Beaver,0,FOOD
...,...,...,...,...,...,...
29923,2061,wholesale trade,5944.337619,Weber,0,WSLE
29924,2062,wholesale trade,5977.005576,Weber,0,WSLE
29925,2063,wholesale trade,6010.644866,Weber,0,WSLE
29926,2064,wholesale trade,6043.334823,Weber,0,WSLE


In [152]:
# check for any rows with missing data after the merge
null_rows_df = gpi_data_with_tdm_codes_df[gpi_data_with_tdm_codes_df.isna().any(axis=1)]

if null_rows_df.size == 0:
    print("No mismatched rows in the merge.")

No mismatched rows in the merge.


In [153]:
tdm_emp_data_df = gpi_data_with_tdm_codes_df.groupby(['county','year','tdm_sector']).agg({'empcount':'sum'}).reset_index()
tdm_emp_data_df

,county,year,tdm_sector,empcount
0,Beaver,2023,AGRI,738.000000
1,Beaver,2023,CONS,278.000000
2,Beaver,2023,FOOD,517.000000
3,Beaver,2023,GVED,810.000000
4,Beaver,2023,HLTH,160.000000
...,...,...,...,...
13712,Weber,2065,MING,169.223049
13713,Weber,2065,OFFI,14763.036832
13714,Weber,2065,OTHR,56307.804748
13715,Weber,2065,RETL,20376.324678


In [154]:
# Check sums from original GPI data and aggregated TDM data
original_total_emp = gpi_employment_by_county_df['empcount'].sum()
aggregated_total_emp = tdm_emp_data_df['empcount'].sum()
print(f"Original total employment: {original_total_emp}")
print(f"Aggregated total employment: {aggregated_total_emp}") 

Original total employment: 131989760.4699922
Aggregated total employment: 131989760.4699922


In [155]:
# export to processed data folder
tdm_emp_data_df.to_csv("data-processed/tdm_employment_by_county.csv", index=False)